# Gold — daily sales KPI

**Layer:** business aggregate consumed by reports / reverse-ETL.

One row per `order_date`: order count, quantity, revenue.


In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("env", "dev")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

if not catalog or not schema:
    raise ValueError("Job parameters catalog and schema are required")

silver_table = f"{catalog}.{schema}.example_orders_silver"
gold_table = f"{catalog}.{schema}.example_orders_gold_daily"


In [ ]:
gold_df = (
    spark.table(silver_table)
    .groupBy("order_date")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("qty").alias("qty_sum"),
        F.round(F.sum("revenue"), 2).alias("revenue"),
    )
    .orderBy("order_date")
)

gold_df.write.mode("overwrite").saveAsTable(gold_table)
print(f"Wrote {gold_df.count()} KPI rows to {gold_table} (env={env})")
gold_df.show(truncate=False)
